# ACDC Final SDF INR Inference and Figures

Inference-only notebook for the signed-distance INR model from `final.ipynb`.
It loads the final SDF checkpoint, extracts ED and ES ACDC contours, runs the
model, and creates figure-ready visualisations. There are no training,
cache-building, or test-set evaluation cells here.


## 1. Setup


In [ ]:
%pip install -q nibabel scikit-image scipy scikit-learn trimesh plotly rtree


In [ ]:
import os, re, math, time, warnings, configparser, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

import torch
import torch.nn as nn
import torch.nn.functional as F

import nibabel as nib
from skimage.measure import find_contours, marching_cubes
from scipy.ndimage import gaussian_filter
from sklearn.neighbors import KDTree
import trimesh

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")


## 2. Configuration


In [ ]:
MODE = "combined"

CFG = dict(
    mode=MODE,
    input_dim=5 if MODE == "combined" else 4,
    latent_dim=256,
    fourier_L=3,
    decoder_hidden=512,
    decoder_layers=8,
    skip_layer=4,
    sphere_r0=0.5,
    decoder_activation="softplus",
    decoder_softplus_beta=100.0,
    decoder_spectral_norm=False,
    tau_min_norm=0.05,
    delta_cap_norm=0.45,
    grid_res=96,
    iso_level=0.0,
    bbox_pad=0.30,
    mc_smooth_sigma=0.0,
    mc_topology_cleanup=True,
    mc_cleanup_smooth_sigma=1.2,
    mc_slice_cleanup=True,
    mc_slice_min_pixels=12,
    mc_min_component_faces=64,
    mc_min_face_frac=0.02,
    mc_component_dist_weight=8.0,
)

CKPT_CANDIDATES = [
    Path("/kaggle/input/models/andrefce/sdf-final-final/pytorch/default/1/inr_sdf_combined_final(1).pt"),
    Path("/kaggle/working/inr_sdf_combined_continued_best.pt"),
    Path("/kaggle/working/inr_sdf_combined_continued_final.pt"),
    Path("inr_sdf_combined_continued_best.pt"),
    Path("inr_sdf_combined_continued_final.pt"),
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p.exists()), CKPT_CANDIDATES[0])

ACDC_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/andrefce/realmri/training"),
    Path("/kaggle/input/realmri/training"),
    Path("/kaggle/input/acdc/training"),
]
ACDC_PATIENT_DIR = Path("/kaggle/input/datasets/andrefce/realmri/training/patient001")

def resolve_default_patient_dir(default_dir):
    if default_dir.exists():
        return default_dir
    for root in ACDC_ROOT_CANDIDATES:
        if root.exists():
            hits = sorted(p for p in root.glob("patient*") if p.is_dir())
            if hits:
                return hits[0]
    return default_dir

ACDC_PATIENT_DIR = resolve_default_patient_dir(ACDC_PATIENT_DIR)
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LBL_BG, LBL_RV, LBL_MYO, LBL_LV = 0, 1, 2, 3
N_PTS_PER_RING = 60
FLIP_Z = True
INFERENCE_GRID_RES = int(CFG["grid_res"])

print(f"Checkpoint: {CKPT_PATH}")
print(f"ACDC patient: {ACDC_PATIENT_DIR}")
print(f"Output dir: {OUT_DIR.resolve()}")
print(f"Grid: {INFERENCE_GRID_RES}^3 | input_dim={CFG['input_dim']} | fourier_L={CFG['fourier_L']}")


## 3. Final SDF Model


In [ ]:
class FourierPE(nn.Module):
    def __init__(self, L=6):
        super().__init__()
        self.L = L
        freqs = 2.0 ** torch.arange(L).float() * math.pi
        self.register_buffer("freqs", freqs)

    @property
    def out_dim(self):
        return 3 + 6 * self.L

    def forward(self, xyz):
        x = xyz.unsqueeze(-1) * self.freqs
        sin = torch.sin(x)
        cos = torch.cos(x)
        return torch.cat([xyz, sin.flatten(-2), cos.flatten(-2)], dim=-1)


class PointNetEncoder(nn.Module):
    def __init__(self, input_dim=4, latent_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(inplace=True),
            nn.Linear(64, 128), nn.ReLU(inplace=True),
            nn.Linear(128, 256), nn.ReLU(inplace=True),
            nn.Linear(256, latent_dim),
        )
        self.proj = nn.Linear(latent_dim * 2, latent_dim)
        nn.init.normal_(self.proj.weight, 0.0, 0.01)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x, mask):
        f = self.mlp(x)
        neg_inf = torch.finfo(f.dtype).min
        tissue = x[:, :, 3]
        endo_mask = mask & (tissue < 0.5)
        epi_mask = mask & (tissue >= 0.5)

        f_endo = f.masked_fill(~endo_mask.unsqueeze(-1), neg_inf)
        f_epi = f.masked_fill(~epi_mask.unsqueeze(-1), neg_inf)
        z_endo = f_endo.max(dim=1).values
        z_epi = f_epi.max(dim=1).values

        f_global = f.masked_fill(~mask.unsqueeze(-1), neg_inf)
        z_global = f_global.max(dim=1).values
        has_endo = endo_mask.any(dim=1, keepdim=True).float()
        has_epi = epi_mask.any(dim=1, keepdim=True).float()
        z_endo = z_endo * has_endo + z_global * (1.0 - has_endo)
        z_epi = z_epi * has_epi + z_global * (1.0 - has_epi)
        return self.proj(torch.cat([z_endo, z_epi], dim=-1))


class INRDecoderSDF(nn.Module):
    def __init__(self, latent_dim=256, fourier_L=6, hidden=512,
                 n_layers=8, skip_layer=4, r0=0.5,
                 delta_init_norm=0.28, delta_cap=None,
                 activation="relu", softplus_beta=100.0,
                 spectral_norm=False):
        super().__init__()
        self.skip_layer = skip_layer
        self.tau_min = float(delta_init_norm)
        self.delta_cap = None if delta_cap is None else float(delta_cap)
        if activation == "softplus":
            beta = float(softplus_beta)
            self._act = lambda x: F.softplus(x, beta=beta, threshold=20.0)
        else:
            self._act = lambda x: F.relu(x, inplace=False)

        in_dim = latent_dim + 3 + 6 * fourier_L
        self.in_proj = nn.Linear(in_dim, hidden)
        self.layers = nn.ModuleList()
        for li in range(n_layers):
            d_in = hidden + (in_dim if li == skip_layer else 0)
            self.layers.append(nn.Linear(d_in, hidden))
        self.head_endo = nn.Linear(hidden, 1)
        self.head_delta = nn.Linear(hidden, 1)

        with torch.no_grad():
            for m in self.layers:
                nn.init.normal_(m.weight, 0.0, math.sqrt(2.0 / m.in_features))
                nn.init.zeros_(m.bias)
            nn.init.normal_(self.head_endo.weight, 0.0, math.sqrt(math.pi) / math.sqrt(hidden))
            nn.init.constant_(self.head_endo.bias, -r0)
            nn.init.normal_(self.head_delta.weight, 0.0, 1e-2)
            if self.delta_cap is None:
                d = max(delta_init_norm - 1e-4, 1e-3)
                self.head_delta.bias.data.fill_(math.log(math.expm1(d)))
            else:
                self.head_delta.bias.data.fill_(0.0)

        if spectral_norm:
            self.in_proj = nn.utils.spectral_norm(self.in_proj)
            self.layers = nn.ModuleList([nn.utils.spectral_norm(m) for m in self.layers])

    def forward(self, z, fxyz):
        B, N, _ = fxyz.shape
        z_exp = z.unsqueeze(1).expand(B, N, -1)
        h_in = torch.cat([z_exp, fxyz], dim=-1)
        h = self._act(self.in_proj(h_in))
        for li, lyr in enumerate(self.layers):
            if li == self.skip_layer:
                h = torch.cat([h, h_in], dim=-1)
            h = self._act(lyr(h))
        f_endo = self.head_endo(h).squeeze(-1)
        raw_d = self.head_delta(h).squeeze(-1)
        if self.delta_cap is None:
            delta = F.softplus(raw_d) + 1e-4
        else:
            delta = self.tau_min + (self.delta_cap - self.tau_min) * torch.sigmoid(raw_d)
        return f_endo, f_endo - delta, delta


class SDFNetwork(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = PointNetEncoder(input_dim=cfg["input_dim"], latent_dim=cfg["latent_dim"])
        self.fourier = FourierPE(L=cfg["fourier_L"])
        self.decoder = INRDecoderSDF(
            latent_dim=cfg["latent_dim"],
            fourier_L=cfg["fourier_L"],
            hidden=cfg["decoder_hidden"],
            n_layers=cfg["decoder_layers"],
            skip_layer=cfg["skip_layer"],
            r0=cfg["sphere_r0"],
            delta_init_norm=cfg["tau_min_norm"],
            delta_cap=cfg.get("delta_cap_norm", None),
            activation=cfg.get("decoder_activation", "relu"),
            softplus_beta=cfg.get("decoder_softplus_beta", 100.0),
            spectral_norm=cfg.get("decoder_spectral_norm", False),
        )

    def encode(self, contour, mask):
        return self.encoder(contour, mask)

    def decode(self, z, query_xyz):
        return self.decoder(z, self.fourier(query_xyz))


def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def strip_dataparallel_prefix(state):
    if not isinstance(state, dict):
        return state
    keys = list(state.keys())
    if keys and all(isinstance(k, str) and k.startswith("module.") for k in keys):
        return {k[len("module."):]: v for k, v in state.items()}
    return state


if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CKPT_PATH}\n"
        "Edit CKPT_PATH in the configuration cell to point at the final.ipynb SDF checkpoint."
    )

ckpt = torch_load_checkpoint(CKPT_PATH)
ckpt_cfg = ckpt.get("cfg", {}) if isinstance(ckpt, dict) else {}
for key in [
    "input_dim", "latent_dim", "fourier_L", "decoder_hidden", "decoder_layers",
    "skip_layer", "sphere_r0", "tau_min_norm", "delta_cap_norm",
    "decoder_activation", "decoder_softplus_beta", "decoder_spectral_norm",
]:
    if key in ckpt_cfg:
        CFG[key] = ckpt_cfg[key]

model = SDFNetwork(CFG).to(DEVICE)
state = ckpt.get("model_state", ckpt) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(strip_dataparallel_prefix(state), strict=True)
model.eval()

print(f"Loaded SDF checkpoint: {CKPT_PATH}")
print(f"Epoch: {ckpt.get('epoch', '?') if isinstance(ckpt, dict) else '?'}")
print(f"Val loss: {ckpt.get('val_loss', float('nan')) if isinstance(ckpt, dict) else float('nan'):.4f}")
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Architecture: input_dim={CFG['input_dim']}, fourier_L={CFG['fourier_L']}, activation={CFG['decoder_activation']}")


## 4. Load ACDC ED and ES Contours


In [ ]:
def resolve_nii_path(p):
    p = Path(p)
    if p.is_file():
        return p
    if p.is_dir():
        for pat in ("*.nii.gz", "*.nii"):
            hits = sorted(p.glob(pat))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"No NIfTI file found at {p}")


def find_ed_es_frames(patient_dir):
    patient_dir = Path(patient_dir)
    if not patient_dir.exists():
        raise FileNotFoundError(f"ACDC patient directory not found: {patient_dir}")
    gt = {}
    for child in sorted(patient_dir.iterdir()):
        m = re.search(r"_frame(\d+)_gt", child.name)
        if m and (child.is_dir() or child.name.endswith((".nii.gz", ".nii"))):
            gt[int(m.group(1))] = child
    if len(gt) < 2:
        raise ValueError(f"Need at least two GT frames in {patient_dir}, found {len(gt)}")

    ed_f = es_f = None
    cfg_path = patient_dir / "Info.cfg"
    if cfg_path.exists():
        parser = configparser.ConfigParser()
        with open(cfg_path, "r", encoding="utf-8") as f:
            parser.read_string("[DEFAULT]\n" + f.read())
        try:
            ed_f = int(parser["DEFAULT"]["ED"])
            es_f = int(parser["DEFAULT"]["ES"])
        except (KeyError, ValueError):
            ed_f = es_f = None

    if ed_f not in gt or es_f not in gt:
        vols = {}
        for frame_no, gp in gt.items():
            vol = nib.load(str(resolve_nii_path(gp))).get_fdata(dtype=np.float32)
            if vol.ndim == 4:
                vol = vol[..., 0]
            vols[frame_no] = int((vol == LBL_LV).sum())
        nonzero = {k: v for k, v in vols.items() if v > 0}
        ed_f = max(nonzero, key=nonzero.get)
        es_f = min(nonzero, key=nonzero.get)
    return gt[ed_f], gt[es_f], ed_f, es_f


def contour_area(pts):
    if len(pts) < 3:
        return 0.0
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def sample_contour(pts2d, n=N_PTS_PER_RING):
    pts2d = np.asarray(pts2d, dtype=np.float32)
    if len(pts2d) < 3:
        return pts2d
    if np.linalg.norm(pts2d[0] - pts2d[-1]) > 1e-6:
        pts2d = np.vstack([pts2d, pts2d[0]])
    d = np.sqrt((np.diff(pts2d, axis=0) ** 2).sum(axis=1))
    total = float(d.sum())
    if total < 1e-6:
        return pts2d[:-1]
    t_old = np.concatenate([[0.0], np.cumsum(d)])
    t_new = np.linspace(0.0, total, int(n), endpoint=False)
    rows = np.interp(t_new, t_old, pts2d[:, 0])
    cols = np.interp(t_new, t_old, pts2d[:, 1])
    return np.column_stack([rows, cols]).astype(np.float32)


def load_and_extract_contours(gt_path):
    real = resolve_nii_path(gt_path)
    nii = nib.load(str(real))
    data = nii.get_fdata().astype(np.int32)
    if data.ndim == 4:
        data = data[..., 0]
    affine = nii.affine
    dz = float(nii.header.get_zooms()[2])
    support = sorted(
        s for s in range(data.shape[2])
        if (data[:, :, s] == LBL_LV).any() or (data[:, :, s] == LBL_MYO).any()
    )

    def vox2world(rows, cols, s):
        vox = np.column_stack([cols, rows, np.zeros(len(rows)), np.ones(len(rows))])
        world = (affine @ vox.T).T
        world[:, 2] = s * dz
        return world[:, :3]

    pts = []
    for s in support:
        seg = data[:, :, s]
        for label, tissue_id in [((seg == LBL_LV), 0.0), (((seg == LBL_MYO) | (seg == LBL_LV)), 1.0)]:
            mask = label.astype(np.uint8)
            if mask.sum() <= 10:
                continue
            contours = find_contours(mask, 0.5)
            if not contours:
                continue
            ring = sample_contour(max(contours, key=contour_area))
            xyz = vox2world(ring[:, 0], ring[:, 1], s)
            pts.append(np.column_stack([xyz, np.full(len(ring), tissue_id, dtype=np.float32)]))

    raw = np.vstack(pts).astype(np.float32)
    xyz, tissue = raw[:, :3], raw[:, 3].astype(np.float32)
    centroid = xyz.mean(axis=0)
    xyz_c = xyz - centroid
    scale = float(np.linalg.norm(xyz_c[:, :2], axis=1).mean())
    if not np.isfinite(scale) or scale < 1e-6:
        scale = float(np.std(xyz_c) + 1e-6)
    xyz_n = (xyz_c / scale).astype(np.float32)
    if FLIP_Z:
        xyz_n[:, 2] = -xyz_n[:, 2]
    return dict(gt_path=real, xyz=xyz_n, tissue=tissue, centroid=centroid.astype(np.float32),
                scale=scale, slices=support, data=data, affine=affine, dz=dz)


ed_gt_path, es_gt_path, ed_frame_no, es_frame_no = find_ed_es_frames(ACDC_PATIENT_DIR)
ed_case = load_and_extract_contours(ed_gt_path)
es_case = load_and_extract_contours(es_gt_path)
ed_case.update(phase="ED", frame=ed_frame_no)
es_case.update(phase="ES", frame=es_frame_no)

for case in (ed_case, es_case):
    print(f"{case['phase']}: frame={case['frame']:02d} | points={len(case['xyz'])} | "
          f"slices={len(case['slices'])} | scale={case['scale']:.2f} mm | path={case['gt_path'].name}")


## 5. SDF Inference Helpers


In [ ]:
def chamfer_mm(pred_verts, gt_verts, scale=1.0):
    """Bidirectional Chamfer distance in mm between two vertex sets."""
    if len(pred_verts) == 0 or len(gt_verts) == 0:
        return float("nan")
    pred = np.asarray(pred_verts, dtype=np.float32)
    gt = np.asarray(gt_verts, dtype=np.float32)
    tree_gt = KDTree(gt)
    tree_pred = KDTree(pred)
    d_p2g, _ = tree_gt.query(pred, k=1)
    d_g2p, _ = tree_pred.query(gt, k=1)
    return float(0.5 * (d_p2g.mean() + d_g2p.mean()) * scale)


def _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val):
    cont = np.column_stack([contour_xyz, tissue_labels]).astype(np.float32)
    if phase_val is not None and cfg["input_dim"] == 5:
        cont = np.column_stack([
            cont, np.full((len(cont), 1), phase_val, dtype=np.float32)
        ])
    cont_t = torch.from_numpy(cont).unsqueeze(0).to(DEVICE)
    mask_t = torch.ones(1, len(cont), dtype=torch.bool, device=DEVICE)
    return cont_t, mask_t


def _slice_residual_mm(mdl, z, contour_xyz, tissue_labels, scale):
    """Mean absolute SDF value on input contour points, in mm."""
    pts = torch.from_numpy(contour_xyz.astype(np.float32)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=False):
        fe, fp, _ = mdl.decode(z, pts)
    fe = fe[0].float().cpu().numpy()
    fp = fp[0].float().cpu().numpy()
    endo_m = tissue_labels == 0
    epi_m = tissue_labels == 1
    res_endo = float(np.abs(fe[endo_m]).mean()) if endo_m.any() else 0.0
    res_epi = float(np.abs(fp[epi_m]).mean()) if epi_m.any() else 0.0
    n = int(endo_m.sum() + epi_m.sum())
    res = (res_endo * endo_m.sum() + res_epi * epi_m.sum()) / max(1, n)
    return float(res * scale)


def _build_grid_and_query(z, mdl, contour_xyz, cfg, grid_res, batch_query):
    """Query the model SDF on a dense grid. No filtering, no field edits."""
    lo = contour_xyz.min(0) - cfg["bbox_pad"]
    hi = contour_xyz.max(0) + cfg["bbox_pad"]
    xs = np.linspace(lo[0], hi[0], grid_res)
    ys = np.linspace(lo[1], hi[1], grid_res)
    zs = np.linspace(lo[2], hi[2], grid_res)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing="ij")
    grid_pts = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=-1).astype(np.float32)

    sdf_e = np.empty(len(grid_pts), np.float32)
    sdf_p = np.empty(len(grid_pts), np.float32)
    dlt = np.empty(len(grid_pts), np.float32)
    with torch.no_grad():
        for s in range(0, len(grid_pts), batch_query):
            chunk = torch.from_numpy(grid_pts[s:s + batch_query]).unsqueeze(0).to(DEVICE)
            with torch.amp.autocast("cuda", enabled=False):
                fe, fp, dl = mdl.decode(z, chunk)
            sdf_e[s:s + batch_query] = fe[0].float().cpu().numpy()
            sdf_p[s:s + batch_query] = fp[0].float().cpu().numpy()
            dlt[s:s + batch_query] = dl[0].float().cpu().numpy()

    shape = (grid_res, grid_res, grid_res)
    voxel = (hi - lo) / (grid_res - 1)
    return sdf_e.reshape(shape), sdf_p.reshape(shape), dlt.reshape(shape), lo, hi, voxel


def _mc_raw(field, lo, voxel, iso_level):
    """Direct marching cubes on the raw model field.

    No Gaussian filtering, no boundary ramp, no padding shell, no component
    selection, no trimesh repair/process, no smoothing. If the raw field
    produces open or fragmented meshes, that is what we want to see.
    """
    field = np.asarray(field, dtype=np.float32)
    if float(field.min()) > iso_level or float(field.max()) < iso_level:
        return trimesh.Trimesh(process=False), True
    try:
        verts, faces, _, _ = marching_cubes(field, level=iso_level, spacing=tuple(voxel))
        verts = verts + lo
        mesh = trimesh.Trimesh(
            vertices=verts.astype(np.float32),
            faces=faces.astype(np.int64),
            process=False,
        )
        return mesh, False
    except Exception:
        return trimesh.Trimesh(process=False), True


@torch.no_grad()
def predict_mesh_sdf(net, contour_xyz, tissue_labels, cfg,
                     grid_res=None, batch_query=131072, phase_val=None,
                     return_latent=False):
    """RAW model inference: model SDF grid -> direct marching cubes."""
    if grid_res is None:
        grid_res = cfg["grid_res"]
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    mdl.eval()

    cont_t, mask_t = _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val)
    z = mdl.encode(cont_t, mask_t)
    sdf_e, sdf_p, dlt, lo, hi, voxel = _build_grid_and_query(
        z, mdl, contour_xyz, cfg, grid_res, batch_query
    )
    endo, deg_e = _mc_raw(sdf_e, lo, voxel, cfg["iso_level"])
    epi, deg_p = _mc_raw(sdf_p, lo, voxel, cfg["iso_level"])

    if return_latent:
        return endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, bool(deg_e or deg_p)
    return endo, epi, sdf_e, sdf_p, dlt


def predict_mesh_sdf_tto(net, contour_xyz, tissue_labels, cfg,
                         grid_res=None, batch_query=131072, phase_val=None,
                         scale=1.0, tto_steps=0, tto_lr=0.0,
                         tto_lambda_z=0.0, tto_lambda_eik=0.0,
                         tto_lambda_sign=0.0, use_tto=False):
    """Compatibility wrapper. TTO is intentionally disabled.

    Visualisation cells still call this name, but it now returns raw model
    output only.
    """
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    out = predict_mesh_sdf(
        net, contour_xyz, tissue_labels, cfg,
        grid_res=grid_res, batch_query=batch_query,
        phase_val=phase_val, return_latent=True,
    )
    endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate = out
    res = _slice_residual_mm(mdl, z, contour_xyz, tissue_labels, scale)
    return dict(
        endo=endo, epi=epi,
        sdf_e=sdf_e, sdf_p=sdf_p, dlt=dlt,
        lo=lo, hi=hi, voxel=voxel,
        z=z, z0=z,
        slice_res_mm_before=res,
        slice_res_mm_after=res,
        tto_history=[],
        degenerate=degenerate,
    )


@torch.no_grad()
def wt_at_endo_vertices(net, contour_xyz, tissue_labels, endo_verts, cfg,
                        phase_val=None, z_override=None):
    """Analytic wall thickness at raw endo-mesh vertices."""
    if len(endo_verts) == 0:
        return np.zeros(0, np.float32)
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    mdl.eval()
    if z_override is None:
        cont_t, mask_t = _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val)
        z = mdl.encode(cont_t, mask_t)
    else:
        z = z_override
    pts = torch.from_numpy(endo_verts.astype(np.float32)).unsqueeze(0).to(DEVICE)
    with torch.amp.autocast("cuda", enabled=False):
        _, _, dl = mdl.decode(z, pts)
    return dl[0].float().cpu().numpy()


print("RAW inference helpers defined: no filtering, repair, component selection, smoothing, or TTO")


In [ ]:
# Topology-cleaned marching-cubes override.
# Converts each predicted SDF sign field into one contour-matched inside
# component before MC. This removes duplicate epi walls and ES bubbles
# without changing the network weights or using mesh repair/cap synthesis.
from scipy.ndimage import binary_fill_holes, distance_transform_edt
from scipy.ndimage import generate_binary_structure, label as ndi_label

_MC_STRUCT = generate_binary_structure(3, 2)
_SLICE_STRUCT = generate_binary_structure(2, 2)

CFG.setdefault('mc_topology_cleanup', True)
CFG.setdefault('mc_cleanup_smooth_sigma', 1.2)
CFG.setdefault('mc_slice_cleanup', True)
CFG.setdefault('mc_slice_min_pixels', 12)
CFG.setdefault('mc_min_component_faces', 64)
CFG.setdefault('mc_component_dist_weight', 8.0)


def _empty_mesh():
    return trimesh.Trimesh(
        vertices=np.empty((0, 3), np.float32),
        faces=np.empty((0, 3), np.int64),
        process=False,
    )


def _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id):
    if contour_xyz is None or tissue_labels is None or tissue_id is None:
        return np.empty((0, 3), np.float32)
    m = np.asarray(tissue_labels) == tissue_id
    if not np.any(m):
        return np.empty((0, 3), np.float32)
    return np.asarray(contour_xyz, dtype=np.float32)[m]


def _select_inside_component(inside, lo, voxel, contour_pts):
    labels, n_comp = ndi_label(inside, structure=_MC_STRUCT)
    if n_comp <= 1:
        return inside.astype(bool, copy=False)

    sizes = np.bincount(labels.ravel()).astype(np.float64)
    sizes[0] = 0.0
    votes = np.zeros_like(sizes)

    if contour_pts is not None and len(contour_pts):
        dims = np.asarray(labels.shape, dtype=np.int64)
        ijk = np.rint((np.asarray(contour_pts) - lo) / voxel).astype(np.int64)
        ijk = np.clip(ijk, 0, dims - 1)
        vote_radius = 2
        for ix, iy, iz in ijk:
            nb = labels[
                max(0, ix - vote_radius):min(dims[0], ix + vote_radius + 1),
                max(0, iy - vote_radius):min(dims[1], iy + vote_radius + 1),
                max(0, iz - vote_radius):min(dims[2], iz + vote_radius + 1),
            ].ravel()
            nb = nb[nb > 0]
            if len(nb):
                lab, cnt = np.unique(nb, return_counts=True)
                votes[lab] += cnt

    if votes.max() > 0:
        score = sizes / max(sizes.max(), 1.0) + 4.0 * votes / votes.max()
    else:
        score = sizes
    keep = int(np.argmax(score[1:]) + 1)
    return labels == keep


def _clean_inside_slice(mask2d, min_pixels=12):
    if int(mask2d.sum()) < int(min_pixels):
        return np.zeros_like(mask2d, dtype=bool)
    labels, n_comp = ndi_label(mask2d, structure=_SLICE_STRUCT)
    if n_comp == 0:
        return np.zeros_like(mask2d, dtype=bool)
    sizes = np.bincount(labels.ravel()).astype(np.int64)
    sizes[0] = 0
    keep = int(np.argmax(sizes[1:]) + 1)
    out = labels == keep
    out = binary_fill_holes(out, structure=_SLICE_STRUCT).astype(bool)
    return out


def _slice_clean_inside(inside, cfg):
    if not cfg.get('mc_slice_cleanup', True):
        return inside.astype(bool, copy=False)
    min_pixels = int(cfg.get('mc_slice_min_pixels', 12))
    out = np.zeros_like(inside, dtype=bool)
    for k in range(inside.shape[2]):
        out[:, :, k] = _clean_inside_slice(inside[:, :, k], min_pixels=min_pixels)
    return out




def _topology_clean_field(field, lo, voxel, iso_level, contour_xyz,
                          tissue_labels, tissue_id, cfg):
    if not cfg.get('mc_topology_cleanup', True):
        return field.astype(np.float32, copy=False)

    inside = np.asarray(field <= iso_level, dtype=bool)
    inside[[0, -1], :, :] = False
    inside[:, [0, -1], :] = False
    inside[:, :, [0, -1]] = False
    if (not inside.any()) or inside.all():
        return field.astype(np.float32, copy=False)

    pts = _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id)
    inside = _select_inside_component(inside, lo, voxel, pts)
    inside = _slice_clean_inside(inside, cfg)
    inside = binary_fill_holes(inside, structure=_MC_STRUCT).astype(bool)
    inside = _slice_clean_inside(inside, cfg)
    inside[[0, -1], :, :] = False
    inside[:, [0, -1], :] = False
    inside[:, :, [0, -1]] = False
    if (not inside.any()) or inside.all():
        return field.astype(np.float32, copy=False)

    outside_dist = distance_transform_edt(~inside, sampling=tuple(voxel))
    inside_dist = distance_transform_edt(inside, sampling=tuple(voxel))
    clean = outside_dist - inside_dist

    sigma = float(cfg.get('mc_cleanup_smooth_sigma', 0.0) or 0.0)
    if sigma > 0:
        clean = gaussian_filter(clean.astype(np.float32), sigma=sigma)
    return clean.astype(np.float32)


def _single_mesh_component(mesh, contour_xyz, tissue_labels, tissue_id, cfg):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return _empty_mesh()
    try:
        parts = list(mesh.split(only_watertight=False))
    except Exception:
        parts = [mesh]
    parts = [p for p in parts if len(p.faces) > 0 and len(p.vertices) > 0]
    if len(parts) <= 1:
        out = parts[0].copy() if parts else _empty_mesh()
        if len(out.vertices):
            out.remove_unreferenced_vertices()
        return out

    pts = _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id)
    diag = float(np.linalg.norm(np.asarray(contour_xyz).max(0) - np.asarray(contour_xyz).min(0))) if contour_xyz is not None and len(contour_xyz) else 1.0
    diag = max(diag, 1e-6)
    max_faces = max(len(p.faces) for p in parts)
    min_faces = max(
        int(cfg.get('mc_min_component_faces', 64)),
        int(max_faces * float(cfg.get('mc_min_face_frac', 0.0) or 0.0)),
    )
    dist_w = float(cfg.get('mc_component_dist_weight', 8.0) or 0.0)

    best_i, best_score = 0, -np.inf
    for i, part in enumerate(parts):
        score = math.log1p(len(part.faces))
        if len(part.faces) < min_faces:
            score -= 2.0
        if len(pts):
            try:
                d, _ = KDTree(np.asarray(part.vertices, dtype=np.float32)).query(pts, k=1)
                score -= dist_w * float(np.median(d)) / diag
            except Exception:
                pass
        if score > best_score:
            best_i, best_score = i, score

    out = parts[best_i].copy()
    out.remove_unreferenced_vertices()
    try:
        out.fix_normals()
    except Exception:
        pass
    return out


def _mc_single_surface(field, lo, voxel, iso_level, contour_xyz=None,
                       tissue_labels=None, tissue_id=None, cfg=None):
    cfg = CFG if cfg is None else cfg
    field = np.nan_to_num(np.asarray(field, dtype=np.float32),
                          nan=1e3, posinf=1e3, neginf=-1e3)

    sigma = float(cfg.get('mc_smooth_sigma', 0.0) or 0.0)
    work = gaussian_filter(field, sigma=sigma).astype(np.float32) if sigma > 0 else field
    work = _topology_clean_field(work, lo, voxel, iso_level,
                                 contour_xyz, tissue_labels, tissue_id, cfg)

    if float(work.min()) > iso_level or float(work.max()) < iso_level:
        return _empty_mesh(), True, work
    try:
        verts, faces, _, _ = marching_cubes(work, level=iso_level, spacing=tuple(voxel))
        mesh = trimesh.Trimesh(
            vertices=(verts + lo).astype(np.float32),
            faces=faces.astype(np.int64),
            process=False,
        )
        mesh = _single_mesh_component(mesh, contour_xyz, tissue_labels, tissue_id, cfg)
        return mesh, len(mesh.faces) == 0, work
    except Exception:
        return _empty_mesh(), True, work


def _mc_raw(field, lo, voxel, iso_level):
    mesh, deg, _ = _mc_single_surface(field, lo, voxel, iso_level, cfg=CFG)
    return mesh, deg


@torch.no_grad()
def predict_mesh_sdf(net, contour_xyz, tissue_labels, cfg,
                     grid_res=None, batch_query=131072, phase_val=None,
                     return_latent=False):
    if grid_res is None:
        grid_res = cfg['grid_res']
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    mdl.eval()

    cont_t, mask_t = _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val)
    z = mdl.encode(cont_t, mask_t)
    raw_e, raw_p, dlt, lo, hi, voxel = _build_grid_and_query(
        z, mdl, contour_xyz, cfg, grid_res, batch_query
    )
    endo, deg_e, sdf_e = _mc_single_surface(
        raw_e, lo, voxel, cfg['iso_level'], contour_xyz, tissue_labels, 0, cfg
    )
    epi, deg_p, sdf_p = _mc_single_surface(
        raw_p, lo, voxel, cfg['iso_level'], contour_xyz, tissue_labels, 1, cfg
    )

    if return_latent:
        return endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, bool(deg_e or deg_p)
    return endo, epi, sdf_e, sdf_p, dlt


def predict_mesh_sdf_tto(net, contour_xyz, tissue_labels, cfg,
                         grid_res=None, batch_query=131072, phase_val=None,
                         scale=1.0, tto_steps=0, tto_lr=0.0,
                         tto_lambda_z=0.0, tto_lambda_eik=0.0,
                         tto_lambda_sign=0.0, use_tto=False):
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    out = predict_mesh_sdf(
        net, contour_xyz, tissue_labels, cfg,
        grid_res=grid_res, batch_query=batch_query,
        phase_val=phase_val, return_latent=True,
    )
    endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate = out
    res = _slice_residual_mm(mdl, z, contour_xyz, tissue_labels, scale)
    return dict(
        endo=endo, epi=epi,
        sdf_e=sdf_e, sdf_p=sdf_p, dlt=dlt,
        lo=lo, hi=hi, voxel=voxel,
        z=z, z0=z,
        slice_res_mm_before=res,
        slice_res_mm_after=res,
        tto_history=[],
        degenerate=degenerate,
    )


print('Topology-cleaned inference helpers active: one contour-matched MC surface per tissue')


## 6. Run Inference


In [ ]:
def infer_case(case, grid_res=INFERENCE_GRID_RES):
    phase_val = None
    if CFG["input_dim"] == 5:
        phase_val = 1.0 if case["phase"] == "ES" else 0.0
    t0 = time.time()
    out = predict_mesh_sdf(
        model, case["xyz"], case["tissue"], CFG,
        grid_res=grid_res, phase_val=phase_val, return_latent=True,
    )
    endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate = out
    wt_norm = wt_at_endo_vertices(
        model, case["xyz"], case["tissue"], endo.vertices,
        CFG, phase_val=phase_val, z_override=z,
    )
    case.update(
        phase_val=phase_val, endo=endo, epi=epi,
        sdf_endo=sdf_e, sdf_epi=sdf_p, delta_grid=dlt,
        latent=z, grid_lo=lo, grid_hi=hi, voxel=voxel,
        degenerate=degenerate, wt_mm=wt_norm * case["scale"],
        elapsed_sec=time.time() - t0,
    )
    return case


print(f"Running final SDF model at grid={INFERENCE_GRID_RES}^3")
ed_case = infer_case(ed_case)
es_case = infer_case(es_case)

def mesh_volume_ml(mesh, scale_mm):
    if mesh is None or len(mesh.faces) == 0:
        return float("nan")
    try:
        return float(abs(mesh.volume) * (scale_mm ** 3) / 1000.0)
    except Exception:
        return float("nan")

summary_df = pd.DataFrame([
    dict(
        phase=case["phase"],
        frame=case["frame"],
        points=len(case["xyz"]),
        slices=len(case["slices"]),
        scale_mm=case["scale"],
        endo_faces=len(case["endo"].faces),
        epi_faces=len(case["epi"].faces),
        endo_volume_ml=mesh_volume_ml(case["endo"], case["scale"]),
        epi_volume_ml=mesh_volume_ml(case["epi"], case["scale"]),
        median_wt_mm=float(np.nanmedian(case["wt_mm"])) if len(case["wt_mm"]) else float("nan"),
        p05_wt_mm=float(np.nanpercentile(case["wt_mm"], 5)) if len(case["wt_mm"]) else float("nan"),
        p95_wt_mm=float(np.nanpercentile(case["wt_mm"], 95)) if len(case["wt_mm"]) else float("nan"),
        seconds=case["elapsed_sec"],
        degenerate=case["degenerate"],
    )
    for case in (ed_case, es_case)
])
display(summary_df.round(3))


## 7. Publication-Style Figure Helpers


In [ ]:
STYLE = {
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Times"],
    "font.size": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

PALETTE = dict(
    endo="#1f77b4", epi="#d62728",
    ed="#2c7fb8", es="#d95f0e",
    contour_endo="#4db8ff", contour_epi="#ff6b6b",
)
SEG_CMAP = ListedColormap([
    (0.00, 0.00, 0.00, 0.00),
    (0.30, 0.50, 1.00, 1.00),
    (0.20, 0.80, 0.30, 1.00),
    (1.00, 0.30, 0.30, 1.00),
])

def save_figure(fig, name):
    png = OUT_DIR / f"{name}.png"
    pdf = OUT_DIR / f"{name}.pdf"
    fig.savefig(png, facecolor=fig.get_facecolor())
    fig.savefig(pdf, facecolor=fig.get_facecolor())
    print(f"Saved {png}")
    print(f"Saved {pdf}")

def plot_faces(mesh, max_faces=12000):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return None, None
    vertices = np.asarray(mesh.vertices, dtype=np.float32)
    faces = np.asarray(mesh.faces, dtype=np.int64)
    if len(faces) > max_faces:
        keep = np.linspace(0, len(faces) - 1, int(max_faces)).astype(np.int64)
        faces = faces[keep]
    return vertices, faces

def add_solid_mesh(ax, mesh, color, alpha=0.85, max_faces=12000):
    vertices, faces = plot_faces(mesh, max_faces=max_faces)
    if vertices is None:
        return
    ax.add_collection3d(Poly3DCollection(vertices[faces], facecolor=color, edgecolor="none", alpha=alpha))

def add_colored_mesh(ax, mesh, values, cmap="viridis", vmin=None, vmax=None, max_faces=14000):
    vertices, faces = plot_faces(mesh, max_faces=max_faces)
    if vertices is None or values is None or len(values) == 0:
        return None
    values = np.asarray(values, dtype=np.float32)
    face_values = values[faces].mean(axis=1)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap_fn = plt.get_cmap(cmap)
    ax.add_collection3d(Poly3DCollection(vertices[faces], facecolors=cmap_fn(norm(face_values)), edgecolor="none", alpha=0.95))
    return plt.cm.ScalarMappable(norm=norm, cmap=cmap_fn)

def set_3d_lim(ax, arrays, pad=0.08):
    pts = [np.asarray(a) for a in arrays if a is not None and len(a)]
    if not pts:
        return
    pts = np.vstack(pts)
    pts = pts[np.isfinite(pts).all(axis=1)]
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    ctr = (lo + hi) * 0.5
    rad = max(float((hi - lo).max()) * (0.5 + pad), 1e-3)
    ax.set_xlim(ctr[0] - rad, ctr[0] + rad)
    ax.set_ylim(ctr[1] - rad, ctr[1] + rad)
    ax.set_zlim(ctr[2] - rad, ctr[2] + rad)
    ax.set_box_aspect([1, 1, 1])

def support_mid_slice(data):
    support = np.where(((data == LBL_LV) | (data == LBL_MYO)).any(axis=(0, 1)))[0]
    return int(support[len(support) // 2]) if len(support) else int(data.shape[2] // 2)

def mesh_to_world(mesh, centroid, scale, flip_z=FLIP_Z):
    if mesh is None or len(mesh.faces) == 0:
        return _empty_mesh()
    vertices = np.asarray(mesh.vertices, dtype=np.float64).copy()
    if flip_z:
        vertices[:, 2] = -vertices[:, 2]
    vertices = vertices * float(scale) + np.asarray(centroid, dtype=np.float64)
    return trimesh.Trimesh(vertices=vertices, faces=mesh.faces.copy(), process=False)


## 8. Figure 1 - ACDC Input and Solid SDF Surfaces


In [ ]:
def figure_acdc_input_and_solids(cases):
    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(16, 7.2), facecolor="white")
        gs = fig.add_gridspec(2, 4, hspace=0.25, wspace=0.15)
        all_vertices = [m.vertices for c in cases for m in (c["endo"], c["epi"]) if len(m.vertices)]
        all_vertices.append(np.zeros((1, 3), dtype=np.float32))

        for r, case in enumerate(cases):
            phase = case["phase"]
            phase_color = PALETTE["ed"] if phase == "ED" else PALETTE["es"]
            xyz, tissue = case["xyz"], case["tissue"]

            ax0 = fig.add_subplot(gs[r, 0])
            z_mid = support_mid_slice(case["data"])
            ax0.imshow(case["data"][:, :, z_mid].T, cmap=SEG_CMAP, vmin=0, vmax=3, origin="lower", interpolation="nearest")
            ax0.set_title(f"{phase} frame {case['frame']} raw ACDC slice z={z_mid}")
            ax0.axis("off")

            ax1 = fig.add_subplot(gs[r, 1])
            ax1.scatter(xyz[tissue == 0, 0], xyz[tissue == 0, 1], c=xyz[tissue == 0, 2], cmap="coolwarm", s=4, alpha=0.85, label="endo")
            ax1.scatter(xyz[tissue == 1, 0], xyz[tissue == 1, 1], c=xyz[tissue == 1, 2], cmap="coolwarm", s=4, alpha=0.45, marker="x", label="epi")
            ax1.set_aspect("equal")
            ax1.set_title(f"{phase} extracted SAX contour rings")
            ax1.set_xticks([])
            ax1.set_yticks([])
            if r == 0:
                ax1.legend(frameon=False, loc="upper right", fontsize=8)

            for c, (elev, azim, title) in enumerate([(20, 35, "solid view"), (82, 0, "top view")], start=2):
                ax = fig.add_subplot(gs[r, c], projection="3d")
                ax.set_facecolor("white")
                ax.set_axis_off()
                ax.view_init(elev=elev, azim=azim)
                add_solid_mesh(ax, case["epi"], PALETTE["epi"], alpha=0.28)
                add_solid_mesh(ax, case["endo"], PALETTE["endo"], alpha=0.92)
                ax.scatter(*xyz[tissue == 0].T, c=PALETTE["contour_endo"], s=2.0, alpha=0.55, depthshade=False)
                ax.scatter(*xyz[tissue == 1].T, c=PALETTE["contour_epi"], s=2.0, alpha=0.35, depthshade=False)
                set_3d_lim(ax, all_vertices + [xyz])
                ax.set_title(f"{phase} {title}", color=phase_color, fontsize=10)

        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - final SDF INR inference on ACDC ED and ES", fontsize=13, fontweight="bold", y=0.99)
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_sdf_inference_preview")
        plt.show()

figure_acdc_input_and_solids([ed_case, es_case])


## 9. Figure 2 - Slice Plane Intersections


In [ ]:
def mesh_section_xy(mesh, z_val):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return []
    try:
        section = mesh.section(plane_origin=[0.0, 0.0, float(z_val)], plane_normal=[0.0, 0.0, 1.0])
    except Exception:
        return []
    if section is None:
        return []
    try:
        return [np.asarray(line)[:, :2] for line in section.discrete if len(line) >= 2]
    except Exception:
        return []

def selected_slice_z(case, n=6):
    z_values = np.sort(np.unique(np.round(case["xyz"][:, 2], 5)))
    if len(z_values) <= n:
        return z_values
    idx = np.linspace(0, len(z_values) - 1, int(n)).round().astype(int)
    return z_values[idx]

def figure_slice_intersections(cases, n_slices=6):
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(len(cases), n_slices, figsize=(2.2 * n_slices, 4.8), facecolor="white", squeeze=False)
        for r, case in enumerate(cases):
            xyz, tissue = case["xyz"], case["tissue"]
            z_round = np.round(xyz[:, 2], 5)
            for c, z_val in enumerate(selected_slice_z(case, n_slices)):
                ax = axes[r, c]
                ax.set_aspect("equal")
                ax.axis("off")
                for line in mesh_section_xy(case["epi"], z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["epi"], lw=1.25, ls="--")
                for line in mesh_section_xy(case["endo"], z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["endo"], lw=1.5)
                sm = z_round == round(float(z_val), 5)
                ax.scatter(xyz[sm & (tissue == 1), 0], xyz[sm & (tissue == 1), 1], s=8, color=PALETTE["contour_epi"], alpha=0.55)
                ax.scatter(xyz[sm & (tissue == 0), 0], xyz[sm & (tissue == 0), 1], s=8, color=PALETTE["contour_endo"], alpha=0.85)
                xy = xyz[sm, :2] if sm.any() else xyz[:, :2]
                ctr = xy.mean(axis=0)
                rad = max(float(np.ptp(xy, axis=0).max()) * 0.65, 1e-3)
                ax.set_xlim(ctr[0] - rad, ctr[0] + rad)
                ax.set_ylim(ctr[1] - rad, ctr[1] + rad)
                ax.set_title(f"{case['phase']} z={float(z_val):+.2f}", fontsize=8)

        fig.legend(handles=[
            plt.Line2D([], [], color=PALETTE["endo"], lw=1.5, label="endo section"),
            plt.Line2D([], [], color=PALETTE["epi"], lw=1.3, ls="--", label="epi section"),
            plt.Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["contour_endo"], markersize=5, label="input endo"),
            plt.Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["contour_epi"], markersize=5, label="input epi"),
        ], loc="lower center", ncol=4, frameon=False, fontsize=9)
        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - mesh section against input SAX contours", fontsize=12, fontweight="bold", y=0.98)
        plt.tight_layout(rect=[0, 0.07, 1, 0.94])
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_sdf_slice_intersections")
        plt.show()

figure_slice_intersections([ed_case, es_case], n_slices=6)


## 10. Figure 3 - Wall Thickness Solid Surface


In [ ]:
def figure_wall_thickness_surface(cases):
    all_wt = np.concatenate([c["wt_mm"][np.isfinite(c["wt_mm"])] for c in cases if len(c["wt_mm"])])
    if len(all_wt) == 0:
        print("No wall-thickness values available.")
        return
    vmin = max(0.0, float(np.percentile(all_wt, 2)))
    vmax = float(np.percentile(all_wt, 98))
    if vmax <= vmin:
        vmax = vmin + 1.0
    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(12, 5.6), facecolor="white")
        sm = None
        all_vertices = [m.vertices for c in cases for m in (c["endo"], c["epi"]) if len(m.vertices)]
        all_vertices.append(np.zeros((1, 3), dtype=np.float32))
        for i, case in enumerate(cases, start=1):
            ax = fig.add_subplot(1, len(cases), i, projection="3d")
            ax.set_facecolor("white")
            ax.set_axis_off()
            ax.view_init(elev=20, azim=35)
            add_solid_mesh(ax, case["epi"], PALETTE["epi"], alpha=0.16)
            sm = add_colored_mesh(ax, case["endo"], case["wt_mm"], cmap="viridis", vmin=vmin, vmax=vmax)
            set_3d_lim(ax, all_vertices)
            ax.set_title(f"{case['phase']} frame {case['frame']}\nmedian WT={np.nanmedian(case['wt_mm']):.1f} mm")
        if sm is not None:
            cbar = fig.colorbar(sm, ax=fig.axes, shrink=0.72, pad=0.02)
            cbar.set_label("Predicted wall thickness (mm)")
        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - SDF decoder wall thickness", fontsize=12, fontweight="bold", y=0.98)
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_sdf_wall_thickness_surface")
        plt.show()

figure_wall_thickness_surface([ed_case, es_case])


## 11. Interactive Solid Model

This Plotly view is a shaded solid surface, not a wireframe. Meshes are
transformed back to world millimetres for inspection.


In [ ]:
%pip install -q plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def mesh3d_trace(mesh, color, opacity, name):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return None
    v = np.asarray(mesh.vertices, dtype=np.float64)
    f = np.asarray(mesh.faces, dtype=np.int64)
    return go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color, opacity=float(opacity), name=name,
        flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.75, specular=0.35, roughness=0.35, fresnel=0.15),
        lightposition=dict(x=200, y=200, z=400),
        showscale=False, hoverinfo="name",
    )

def world_meshes(case):
    return dict(
        endo=mesh_to_world(case["endo"], case["centroid"], case["scale"], FLIP_Z),
        epi=mesh_to_world(case["epi"], case["centroid"], case["scale"], FLIP_Z),
    )

def show_solid_side_by_side(cases):
    fig = make_subplots(
        rows=1, cols=len(cases),
        specs=[[{"type": "scene"} for _ in cases]],
        subplot_titles=[f"{c['phase']} frame {c['frame']}" for c in cases],
        horizontal_spacing=0.02,
    )
    for col, case in enumerate(cases, start=1):
        wm = world_meshes(case)
        for tr in [
            mesh3d_trace(wm["epi"], PALETTE["epi"], 0.30, f"{case['phase']} epi"),
            mesh3d_trace(wm["endo"], PALETTE["endo"], 0.95, f"{case['phase']} endo"),
        ]:
            if tr is not None:
                fig.add_trace(tr, row=1, col=col)
    scene = dict(
        xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
        aspectmode="data", bgcolor="white",
        xaxis=dict(backgroundcolor="white", gridcolor="#e5e5e5"),
        yaxis=dict(backgroundcolor="white", gridcolor="#e5e5e5"),
        zaxis=dict(backgroundcolor="white", gridcolor="#e5e5e5"),
        camera=dict(eye=dict(x=1.55, y=1.55, z=1.05)),
    )
    fig.update_layout(
        title=dict(text=f"{ACDC_PATIENT_DIR.name} - interactive solid SDF reconstruction", x=0.5),
        height=660, width=1200, margin=dict(l=0, r=0, t=70, b=0), showlegend=False,
    )
    for i in range(1, len(cases) + 1):
        fig.update_layout(**{f"scene{i if i > 1 else ''}": scene})
    fig.show()

show_solid_side_by_side([ed_case, es_case])


## 12. Optional Mesh Export


In [ ]:
EXPORT_PLY = False

def export_world_meshes(cases):
    saved = []
    for case in cases:
        wm = world_meshes(case)
        for name, mesh in wm.items():
            if len(mesh.faces) == 0:
                continue
            path = OUT_DIR / f"{ACDC_PATIENT_DIR.name}_{case['phase']}_{name}_sdf_final_world_mm.ply"
            mesh.export(path)
            saved.append(path)
    return saved

if EXPORT_PLY:
    for path in export_world_meshes([ed_case, es_case]):
        print(f"Saved {path}")
else:
    print("Set EXPORT_PLY = True to export predicted ED/ES endo and epi surfaces as world-mm PLY files.")
